In [ ]:
import json
import pathlib

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from pprint import pprint

import src.utils as utils
import src.visuals as visual

from src.models import GINO
from src.loss import NavierStokesLoss
from src.dataloader import load_data, gen_state_dataloaders
from src.train import train_model

In [ ]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# TRENING: uzorkovane (frac_1) tacke — isti split kao PINN heldout eksperiment.
# TEST:    puni CFD snapshotovi (cijela mreza po stanju) za gustu evaluaciju.
# NB: GINO je operator; frac_1 daje malo tacaka po (re, time) stanju (medijan ~6),
#     pa je po-stanju polje rijetko. knn_indices klampuje k=min(k, N) tako da radi,
#     ali kvalitet operatora zavisi od gustine tacaka u stanju.
TRAIN_DATA_PATH = pathlib.Path("data/sampled_data/frac_1")
TRAIN_FILE_NAME = "data"

TEST_DATA_PATH = pathlib.Path("data/raw_data")
TEST_FILE_NAME = "full_data_v2"

INPUT_COL_NAMES = ["time", "re", "x", "y"]
TARGET_COL_NAMES = ["U_x", "U_y", "p"]

BOX = {
    "x_min": 0.15,
    "x_max": 0.85,
    "y_min": 0.15,
    "y_max": 0.85,
}

EPOCHS = 500
LEARNING_RATE = 1e-3

PHYSICS_WEIGHTS = [
    0.0,
    0.01,
    0.05,
    0.1,
    0.5,
    1.0
]

LR_FACTOR = 0.5
LR_PATIENCE = 15

# Arhitektura GINO-a se cita iz config-a (physics_weight se gazi po iteraciji petlje).
CFG = yaml.safe_load(open("config/architecture/gino.yaml"))
GROUP_COLS = CFG["training"]["group_cols"]

print(f"Train data: {TRAIN_DATA_PATH} ({TRAIN_FILE_NAME})")
print(f"Test data:  {TEST_DATA_PATH} ({TEST_FILE_NAME})")
print(f"Epochs: {EPOCHS}")
print(f"Physics weights: {PHYSICS_WEIGHTS}")
print(f"Held-out box: {BOX}")
print(f"Group cols (jedno stanje): {GROUP_COLS}")

In [ ]:
train_df, valid_df, _ = load_data(TRAIN_DATA_PATH, TRAIN_FILE_NAME)
_, _, test_df = load_data(TEST_DATA_PATH, TEST_FILE_NAME)

print(f"Train samples:      {len(train_df):,}")
print(f"Validation samples: {len(valid_df):,}")
print(f"Test samples:       {len(test_df):,}")

train_re = list(map(int, sorted(train_df["re"].unique())))
valid_re = list(map(int, sorted(valid_df["re"].unique())))
test_re = list(map(int, sorted(test_df["re"].unique())))

time_steps = list(map(float, sorted(train_df["time"].unique())))

print(f"\nTraining Reynolds numbers:   {train_re}")
print(f"Validation Reynolds numbers: {valid_re}")
print(f"Test Reynolds numbers:       {test_re}")

print(f"\nTime steps: {time_steps}")

In [ ]:
# Izbacujemo tacke UNUTAR boxa iz svakog trening stanja (held-out region).
# Stanja ostaju (samo im nedostaje unutrasnjost); operator i dalje vidi ostatak polja.
_, train_out = utils.split_by_box(train_df, BOX)

test_inside, test_outside = utils.split_by_box(test_df, BOX)

print(f"Original train samples: {len(train_df):,}")
print(f"Train after held-out:   {len(train_out):,}")
print(f"Removed from train:     {len(train_df) - len(train_out):,}")

print(f"\nValidation samples:     {len(valid_df):,}")

print(f"\nTest samples:")
print(f"  Inside box:           {len(test_inside):,}")
print(f"  Outside box:          {len(test_outside):,}")

removed_pct = 100 * (len(train_df) - len(train_out)) / len(train_df)

print(f"\nRemoved from training:  {removed_pct:.2f}%")

In [ ]:
mean = train_out.mean()
std = train_out.std()

train_df_norm = utils.normalize_data(train_out, mean, std)
valid_df_norm = utils.normalize_data(valid_df, mean, std)

In [ ]:
# Per-state loaderi: svaki batch je jedno (re, time) stanje (cijelo polje bez held-out boxa).
train_dataloader, valid_dataloader, _ = gen_state_dataloaders(
    train_df_norm,
    valid_df_norm,
    valid_df_norm,
    INPUT_COL_NAMES,
    TARGET_COL_NAMES,
    group_cols=GROUP_COLS,
)

print(f"Train states:      {len(train_dataloader)}")
print(f"Validation states: {len(valid_dataloader)}")

In [ ]:
def evaluate_gino_regions(
    model,
    df,
    input_cols,
    target_cols,
    mean,
    std,
    device,
    box,
    group_cols=("re", "time"),
):
    """Per-state GINO evaluacija razlozena po regionu.

    Model se poziva na CIJELOM (re, time) polju (operator mora vidjeti cijelu geometriju),
    a metrike se onda racunaju posebno UNUTAR i IZVAN held-out boxa, na originalnoj skali.
    Vraca {region: (n_points, metrics_dict)} kompatibilno sa utils.flatten_metrics.
    """
    model.eval()

    im = mean[input_cols].to_numpy(np.float32)
    istd = std[input_cols].to_numpy(np.float32)
    tm = mean[target_cols].to_numpy(np.float32)
    tstd = std[target_cols].to_numpy(np.float32)

    buckets = {
        "inside_box": {"pred": [], "true": []},
        "outside_box": {"pred": [], "true": []},
    }

    for _, g in df.groupby(list(group_cols), sort=False):
        x = (g[input_cols].to_numpy(np.float32) - im) / istd
        xt = torch.tensor(x, dtype=torch.float32, device=device)

        with torch.no_grad():
            pred = model(xt).cpu().numpy() * tstd + tm

        true = g[target_cols].to_numpy(np.float32)

        inside = (
            (g["x"] >= box["x_min"]) & (g["x"] <= box["x_max"]) &
            (g["y"] >= box["y_min"]) & (g["y"] <= box["y_max"])
        ).to_numpy()

        buckets["inside_box"]["pred"].append(pred[inside])
        buckets["inside_box"]["true"].append(true[inside])
        buckets["outside_box"]["pred"].append(pred[~inside])
        buckets["outside_box"]["true"].append(true[~inside])

    out = {}
    for region, d in buckets.items():
        y_pred = np.concatenate(d["pred"])
        y_true = np.concatenate(d["true"])

        metrics = {"all": utils.compute_metrics(y_true, y_pred)}
        for i, col in enumerate(target_cols):
            metrics[col] = utils.compute_metrics(y_true[:, i], y_pred[:, i])

        out[region] = (len(y_true), metrics)

    return out

In [ ]:
results = []

for c_physics in PHYSICS_WEIGHTS:
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    print("\n" + "=" * 80)
    print(f"Training GINO with c_physics = {c_physics}")
    print("=" * 80)

    tag = f"cphys_{c_physics:g}"

    run_dir = utils.create_run_directory(label=f"gino_heldout_{tag}")

    config = {
        "experiment": "gino_heldout_region",
        "model": "gino",
        "physics_weight": c_physics,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "box": BOX,
        "gino": CFG["gino"],
        "train_states": len(train_dataloader),
        "valid_states": len(valid_dataloader),
        "train_samples": len(train_out),
        "valid_samples": len(valid_df),
        "test_inside_samples": len(test_inside),
        "test_outside_samples": len(test_outside),
    }

    with open(run_dir / "config.json", "w") as f:
        json.dump(config, f, indent=4)

    model = GINO.from_config(CFG).to(device)

    criterion = NavierStokesLoss(
        c_physics,
        mean,
        std,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
    )

    history = train_model(
        model=model,
        train_dataloader=train_dataloader,
        valid_dataloader=valid_dataloader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=EPOCHS,
        run_dir=run_dir,
        checkpoint=None,
        physics_loss=True,
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "history.csv", index=False)

    fig, _ = utils.plot_training_history(
        history_df,
        output_path=run_dir / "losses.png",
        title=f"GINO held-out (c_physics={c_physics})",
    )
    plt.close(fig)

    model.eval()

    region_eval = evaluate_gino_regions(
        model=model,
        df=test_df,
        input_cols=INPUT_COL_NAMES,
        target_cols=TARGET_COL_NAMES,
        mean=mean,
        std=std,
        device=device,
        box=BOX,
        group_cols=GROUP_COLS,
    )

    metrics = []

    for region_name, (n_points, region_metrics) in region_eval.items():
        row = {
            "tag": tag,
            "c_physics": c_physics,
            "region": region_name,
            "n_points": n_points,
        }

        row.update(utils.flatten_metrics(region_metrics))

        metrics.append(row)
        results.append(row)

    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(run_dir / "metrics.csv", index=False)

    print(metrics_df[["region", "R2_all", "MAE_all", "RMSE_all"]])

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["c_physics", "region"]
).reset_index(drop=True)

results_df.to_csv("gino_heldout_results.csv", index=False)

display(results_df)